# 06 Subhalo comparison

Collects the test scores of the subhalo runs into one table (MCC by run and spacing), and checks that the FiLM models actually use the spacing they are given: each is scored on the spacing-10 field while being told spacings from 3 to 10. If the scores barely change, the conditioning does nothing.

No training here. Runtime: **L4 GPU** with **High-RAM** (the sweep predicts the full box several times).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Test MCC by run and spacing

In [ ]:
import json
from pathlib import Path
import pandas as pd

root = Path('/content/drive/MyDrive/deepvoid_runs')
runs = ['subhalo_A_scratch_L10', 'subhalo_B_finetune_L10', 'subhalo_C_mixed',
        'subhalo_C_film', 'subhalo_C_film_curriculum']
rows = {}
for run_name in runs:
    path = root / run_name / 'test_scores.json'
    if not path.exists():
        print('not finished:', run_name)
        continue
    scores = json.load(open(path))
    rows[run_name] = {name.rsplit('/', 1)[-1]: s['mcc'] for name, s in scores.items()}
mcc = pd.DataFrame(rows).T[['L3', 'L5', 'L7', 'L10']]
print(mcc.round(3))

## 2. Spacing sweep for the FiLM runs

Takes a few minutes per run. The true spacing is 10.

In [ ]:
from deepvoid.train import spacing_sweep

told = [3, 5, 7, 10]
sweeps = {}
for run_name in ['subhalo_C_film', 'subhalo_C_film_curriculum']:
    if (root / run_name / 'best.keras').exists():
        result = spacing_sweep(root / run_name, 'tng300-3/subhalos/L10', told)
        sweeps[run_name] = {s: r['mcc'] for s, r in result.items()}
print(pd.DataFrame(sweeps).round(3))

Paste both tables back to Claude.